In [ ]:
from pygom.model.ode_variable import State, Parameter, DerivedParameter, CallableParameter
import pytest
import numpy as np
from sympy import Symbol

# Test Variable Objects

## 1. CallableParameter 

In [2]:
def test_dict_or_tuple(dist, params):
    """
    Check if callable param can be defined via dict or tuple
    """
    x = CallableParameter(
        (dist, params)
    )

    y = CallableParameter(
        (dist, tuple(params.values()))
    )

def test_same_seed(x):
    """
    Test that same seed results in same random draws
    for random process, x
    """
    rng1 = np.random.default_rng(123)
    rng2 = np.random.default_rng(123)

    assert x(rng1) == x(rng2)

def test_different_seed(x):
    """
    Test that different seed results in different random draws
    for random process, x:
    
    TODO: Technically not guaranteed, but failure very unlikely for
          continuous distributions.
    """
    rng1 = np.random.default_rng(123)
    rng2 = np.random.default_rng(456)

    assert x(rng1) != x(rng2)

def test_continuous_output(x, rng):
    """
    Test that continuous output is finite and scalar
    """
    sample = x(rng)

    assert np.isfinite(sample)
    assert np.isscalar(sample)

def test_discrete_output(x, rng):
    """
    Test that discrete output is finite and integer
    """
    sample = x(rng)

    assert np.isfinite(sample)
    assert isinstance(sample, (int, np.integer))

### 1.1.1 Test a few numpy distributions

pygom/src/pygom/utilR/distn.py basically wraps predefined distributions

If we commit to using random numpy, then we can just declare the function name and kwargs

Apply the above tests to selection of distributions

In [3]:
# Continuous
numpy_test_dists_continuous = {
    'gamma': {'shape': 2.0, 'scale':3.0},
    'normal': {'loc': 0.0, 'scale': 1.0},
    'exponential': {'scale': 10.0},
    'uniform': {'low': 0.0, 'high': 10.0},
    'beta': {'a': 1.0, 'b': 1.0}
}

# Discrete
numpy_test_dists_discrete = {
    'poisson': {'lam': 3.0},
    'binomial': {'n': 10, 'p': 0.5},
    'negative_binomial': {'n': 10, 'p': 0.5}
}

for dist, args in numpy_test_dists_continuous.items():
    test_dict_or_tuple(dist, args)

    x = CallableParameter(
        (dist, args)
    )

    test_same_seed(x)
    test_different_seed(x)
    test_continuous_output(x, np.random.default_rng(1234))

for dist, args in numpy_test_dists_discrete.items():
    test_dict_or_tuple(dist, args)

    x = CallableParameter(
        (dist, args)
    )

    test_same_seed(x)
    test_discrete_output(x, np.random.default_rng(1234))

### 1.1.1 Test a custom distribution

The user can also define their own random number generation process (as long as they consist of numpy functions)

In [4]:
def custom_sampler(rng, lo, hi):
    """
    Random variable is the sum of 2 random variables ~ U(lo, hi)
    """
    i1 = rng.uniform(low=lo, high=hi)
    i2 = rng.uniform(low=lo, high=hi)

    return i1 + i2

dist = custom_sampler
params = {'lo': 5, 'hi': 10}

test_dict_or_tuple(dist, params)

x = CallableParameter(
    (dist, params)
)

test_same_seed(x)
test_different_seed(x)
test_continuous_output(x, np.random.default_rng(1234))

## 2. State

## 2.1 Legitimate ways to define  

In [ ]:
def check_state_build():
    """
    Check that the only way to build a State is via a string
    """

    id = 'x'
    state = State(ID=id)

    assert state.ID == id
    assert state.symbol == Symbol(id, real=True)

    invalid_defs = [
        Symbol('x'),
        1
    ]

    for invalid_def in invalid_defs:
        with pytest.raises(TypeError):
            state = State(invalid_def)

check_state_build()

## 2.2 Check numeric values

In [24]:
def test_numeric_limits():
    """
    Test numeric value checks
    """
    # Default limits (0, inf)
    state = State(ID = 'x', initial_value=5)

    with pytest.raises(ValueError):
        state = State(ID = 'x', initial_value=-1)

    # Change limits:
    state.limits = (-10, 10)

    state.initial_value = -1
    with pytest.raises(ValueError):
        state.initial_value = 100

    # Also check that states cannot have value attributes
    with pytest.raises(AttributeError):
        state.value

test_numeric_limits()

AttributeError: 'State' object has no attribute 'value'

# 3. Parameter

## 3.1 Legitimate ways to define

In [25]:
def check_param_build():
    """
    Check that the only way to build a Param is via a string
    """

    id = 'x'
    param = Parameter(ID=id)

    assert param.ID == id
    assert param.symbol == Symbol(id, real=True)

    # Also check that params have value attributes
    param.value

    invalid_defs = [
        Symbol('x'),
        1
    ]

    for invalid_def in invalid_defs:
        with pytest.raises(TypeError):
            param = Parameter(invalid_def)

check_param_build()

## 3.2 NoneType Value

Asking for realisations does not change the value

In [16]:
param = Parameter(
    ID='alpha'
)
assert param.source is None
assert param.value is None
param.realise()
assert param.value is None

## 3.2 Fixed value

Asking for realisations does not change the value

In [17]:
val = 3.14

param = Parameter(
    ID='alpha',
    value = val
)

assert param.source==val
assert param.value==val
param.realise()
assert param.value==val

## 3.3 Random value

Asking for realisations changes the value

In [20]:
param = Parameter(
    ID='alpha',
    value=CallableParameter( ('exponential', {'scale': 3.0}) )
)
assert callable(param.source)
assert param.value is None

rng = np.random.default_rng(123)
for _ in range(10):
    param.realise(rng)
    print(f"Value: {param.value}")

Value: 1.7909174853679555
Value: 0.35109318314500637
Value: 0.7553840984797152
Value: 0.9238963364014241
Value: 0.2754079807893665
Value: 7.054862566622669
Value: 12.865405052339664
Value: 0.46389872323759207
Value: 1.49315724681844
Value: 2.478856924601381


# 4. Derived parameters

## 4.1 Legitimate ways to define

In [26]:
def check_derived_param_build():
    """
    Check that the only way to build a Derived Param is via a string
    and a compulsory string expression
    """

    id = 'x'
    expr = 'a+b+c'
    derived_param = DerivedParameter(
        ID=id,
        string_expression=expr
    )

    assert derived_param.ID == id
    assert derived_param.symbol == Symbol(id, real=True)
    assert derived_param.string_expression == expr

    # Also check that derived params cannot have value attributes
    with pytest.raises(AttributeError):
        derived_param.value

    with pytest.raises(TypeError):
        param = DerivedParameter(ID=id)

check_derived_param_build()

# 4. Stores

In [27]:
from pygom.model.ode_utils.variable_store import ParameterStore

parameter_list = ['alpha', 'beta', 'gamma']

parameter_store = ParameterStore()
parameter_store.add(parameter_list, {})

parameter_store.values = {
    'alpha': 1,
    'beta': 1
}

print(parameter_store.values)
parameter_store.all_values_set

AttributeError: type object 'VariableStore' has no attribute 'values'

In [ ]:
from pygom.model.ode_utils.variable_store import ParameterStore

parameter_list = ['alpha', 'beta', 'gamma']

rng = np.random.default_rng(123)

parameter_store = ParameterStore(rng=rng)
parameter_store.add(parameter_list, {})

parameter_store.values = {
    'alpha': 3.14,
    'beta': ('exponential', {'scale': 10.0}),
    'gamma': ('exponential', {'scale': 10.0})
}

In [ ]:
print(parameter_store.values)
print(parameter_store.values)
parameter_store.new_realisation()
print(parameter_store.values)
print(parameter_store.values)
parameter_store.new_realisation()
print(parameter_store.values)
print(parameter_store.values)

[3.14, 5.969724951226518, 1.1703106104833545]
[3.14, 5.969724951226518, 1.1703106104833545]
[3.14, 2.517946994932384, 3.079654454671414]
[3.14, 2.517946994932384, 3.079654454671414]
[3.14, 0.9180266026312217, 23.516208555408898]
[3.14, 0.9180266026312217, 23.516208555408898]


In [ ]:

# def test_default_symbol_created():
#     var = ODEVariable("x")
#     assert str(var.symbol) == "x"

# # def test_symbol_must_be_sympy_symbol():
# #     with pytest.raises(InputError):
# #         ODEVariable("x", symbol="x")

# def test_string_equality():
#     assert ODEVariable("x") == "x"

# def test_symbol_equality():
#     assert ODEVariable("x") == sympy.Symbol("x")

# def test_different_ids_not_equal():
#     assert ODEVariable("x") != ODEVariable("y")

# def test_repr():
#     repr(ODEVariable("x"))